Task 1
Scrape weather information from an HTML file and process it using Python and BeautifulSoup.

<h4>5-Day Weather Forecast</h4>
<table>
    <thead>
        <tr>
            <th>Day</th>
            <th>Temperature</th>
            <th>Condition</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>Monday</td>
            <td>25°C</td>
            <td>Sunny</td>
        </tr>
        <tr>
            <td>Tuesday</td>
            <td>22°C</td>
            <td>Cloudy</td>
        </tr>
        <tr>
            <td>Wednesday</td>
            <td>18°C</td>
            <td>Rainy</td>
        </tr>
        <tr>
            <td>Thursday</td>
            <td>20°C</td>
            <td>Partly Cloudy</td>
        </tr>
        <tr>
            <td>Friday</td>
            <td>30°C</td>
            <td>Sunny</td>
        </tr>
    </tbody>
</table>
Assume you are given the following HTML structure (you can save it as weather.html):

<!DOCTYPE html>
<html>
<head>
    <title>Weather Forecast</title>
</head>
<body>
    <h4>5-Day Weather Forecast</h4>
    <table>
        <thead>
            <tr>
                <th>Day</th>
                <th>Temperature</th>
                <th>Condition</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Monday</td>
                <td>25°C</td>
                <td>Sunny</td>
            </tr>
            <tr>
                <td>Tuesday</td>
                <td>22°C</td>
                <td>Cloudy</td>
            </tr>
            <tr>
                <td>Wednesday</td>
                <td>18°C</td>
                <td>Rainy</td>
            </tr>
            <tr>
                <td>Thursday</td>
                <td>20°C</td>
                <td>Partly Cloudy</td>
            </tr>
            <tr>
                <td>Friday</td>
                <td>30°C</td>
                <td>Sunny</td>
            </tr>
        </tbody>
    </table>

</body>
</html>
1. Parse the HTML File:

Load the weather.html file using BeautifulSoup and extract the weather forecast details.
2. Display Weather Data:

Print the day, temperature, and condition for each entry in the forecast.
3. Find Specific Data:

Identify and print the day(s) with:
The highest temperature.
The "Sunny" condition.
4. Calculate Average Temperature:

Compute and print the average temperature for the week.

In [ ]:
from bs4 import BeautifulSoup
with open("weather.html", "r", encoding = "utf-8") as f:
    day_count = 0
    soup = BeautifulSoup(f, "html.parser")
    #days and info
    rows = soup.find("tbody").find_all("tr")
    for row in rows:
        cells = row.find_all("td")
        for cell in cells:
            print(cell.text.strip())
            day_count +=1

print(soup.get_text())

sunny_days = []
highest_temp = None
highest_day = None
total_temp = 0
for row in rows:
    cells = row.find_all("td")
    day = cells[0].text
    temp = float(cells[1].text.replace("°C", ""))
    condition = cells[2].text

    if condition == "Sunny":
        sunny_days.append(day)

    if highest_temp is None or temp > highest_temp:
        highest_temp = temp
        highest_day = day

    total_temp +=temp
    average_temp = total_temp / day_count
    average_temp = round(average_temp, 2)

print(f"Sunny days: {sunny_days}")
print(f"Highest temperature: {highest_temp} + °C on {highest_day}")
print(f"Average temperature : {average_temp} °C")



Task 2
Scrape job listings from the website https://realpython.github.io/fake-jobs and store the data into an SQLite database.

1. Scraping Requirements:

Extract the following details for each job listing:
Job Title
Company Name
Location
Job Description
Application Link

2. Data Storage:

Store the scraped data into an SQLite database in a table named jobs.

3. Incremental Load:

Ensure that your script performs incremental loading:
Scrape the webpage and add only new job listings to the database.
Avoid duplicating entries. Use Job Title, Company Name, and Location as unique identifiers for comparison.

4. Update Tracking:

Add functionality to detect if an existing job listing has been updated (e.g., description or application link changes) and update the database record accordingly.

5. Filtering and Exporting:

Allow filtering job listings by location or company name.
Write a function to export filtered results into a CSV file.

In [ ]:
from bs4 import BeautifulSoup
import requests
url = "https://realpython.github.io/fake-jobs"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

jobs = soup.find_all("div", class_ = "card-content")
job_list= []
for job in jobs:
    title = job.find("h2", class_ = "title")
    company = job.find("h3", class_ = "subtitle")
    location = job.find("p", class_ = "location")
    link = job.find("a", string = "Apply")["href"]

    job_page = requests.get(link)
    desc = BeautifulSoup(job_page.text, "html.parser")
    description = desc.find("div", class_ = "content").find("p").text

    job_list.append({
        "title" : title.text,
        "company" : company.text,
        "location" : location.text.strip(),
        "description" : description,
        "link" : link   
    })

import sqlite3

with sqlite3.connect("jobs.db") as c:
    cursor = c.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS "Jobs"(
        "title" TEXT,
        "company" TEXT,
        "location" TEXT,
        "description" TEXT,
        "link" TEXT
    )  
""")
    c.commit()

    for job in job_list:
        cursor.execute("SELECT * FROM Jobs WHERE title = ? AND company = ? AND location = ?", (job["title"], job["company"], job["location"]))

        existing = cursor.fetchone()

        if existing is None:
            cursor.execute("INSERT INTO Jobs VALUES (?, ?, ?, ?, ?)", (job["title"], job["company"], job["location"], job["description"], job["link"]))
        else:
            cursor.execute("UPDATE Jobs SET description = ?, link = ? WHERE title =? AND company = ? AND location = ?", 
                           (job["description"], job["link"], job["title"], job["company"], job["location"]))

    c.commit()


def filtering_jobs(location = None,company = None):
    import sqlite3
    with sqlite3.connect("jobs.db") as c:
        cursor = c.cursor()

        if location:
            cursor.execute("SELECT * FROM Jobs WHERE location = ?", (location,))
        elif company:
            cursor.execute("SELECT * FROM Jobs WHERE company = ?", (company,))
        else:
            cursor.execute("SELECT * FROM Jobs")
        result = cursor.fetchall()
        return result

filtering_jobs(location = "Hullview, AA")


def convert_to_csv(jobs, filename):
    import csv
    with open(filename, 'w', newline = '') as c:
        writer = csv.writer(c)
        writer.writerow(["Title", "Company", "Location", "Description", "Link"])
        for job in jobs:
            writer.writerow(job)
        print(f"Exported {len(jobs)} job(s) to {filename}")



jobs = filtering_jobs(location = "Hullview, AA")
convert_to_csv(jobs, 'jobs.csv')




Task 3
You are tasked with scraping laptop data from the "Laptops" section of the [Demoblaze website](https://www.demoblaze.com/) and storing the extracted information in JSON format.

Steps:

1. Navigate to the Website:

Visit the [Demoblaze homepage](https://www.demoblaze.com/).
Click on the Laptops section to view the list of available laptops.

2. Navigate to the Next Page:

After reaching the Laptops section, locate and click the Next button to navigate to the next page of laptop listings.

3. Data to Scrape: For each laptop on the page, scrape the following details:

Laptop Name
Price
Description

4. Data Storage:

Save the extracted information in a structured JSON format with fields like:
``json [ { "name": "Laptop Name", "price": "Laptop Price", "description": "Laptop Description" }, ... ] ``

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

driver = webdriver.Chrome()
driver.get("https://www.demoblaze.com/")
time.sleep(5)

#clicking laptop section
laptops_link = driver.find_element(By.LINK_TEXT, "Laptops")
laptops_link.click()
time.sleep(5)

laptops = driver.find_elements(By.CLASS_NAME, "card-block")
laptops_list = []
for laptop in laptops:
    name = laptop.find_element(By.CLASS_NAME, "card-title")
    price = laptop.find_element(By.XPATH, ".//h5")
    description = laptop.find_element(By.CLASS_NAME, "card-text")

    laptops_list.append({
        "name": name.text,
        "price": price.text,
        "description": description.text
    })


next_button = driver.find_element(By.ID, "next2")
next_button.click()
time.sleep(5)

more_laptops = driver.find_elements(By.CLASS_NAME,"card-block")
for laptop in more_laptops:
    name = laptop.find_element(By.CLASS_NAME, "card-title")
    price = laptop.find_element(By.XPATH, ".//h5")
    description = laptop.find_element(By.CLASS_NAME, "card-text")

    laptops_list.append({
        "name": name.text,
        "price": price.text,
        "description": description.text
    })

import json
with open("laptops.json", "w") as f:
    json.dump(laptops_list,f, indent = 2)

driver.quit()